# 01 — 数据下载与存储

**负责人：Alex**  
**目的：** 下载 8 个 LeRobot 数据集，合并后存到 Google Drive，供组员共用

运行环境：Google Colab（推荐）或本地

---

## 数据来源说明

我们使用 **LeRobot**（Hugging Face，2024）数据集，共 **8 个子集、251,802 帧**。

### 为什么选 LeRobot 而不是 Open X-Embodiment？

最初考虑 Google 的 Open X-Embodiment（百万条轨迹，154 GB），但实际测试发现：
```
RuntimeError: Dataset scripts are no longer supported
```
该数据集使用自定义加载脚本，与 HuggingFace datasets 4.x 不兼容，无法使用。

LeRobot 使用标准 Parquet 格式，完全兼容新版 datasets 库，且提供统一的列名规范，非常适合跨数据集分析。

### 数据集分组

| 分组 | 数据集 | 机构 | 帧数 |
|------|--------|------|------|
| **仿真** | `lerobot/pusht` | — | 25,650 |
| **仿真** | `lerobot/xarm_lift_medium` | — | 20,000 |
| **仿真** | `lerobot/xarm_lift_medium_replay` | — | 20,000 |
| **仿真** | `lerobot/xarm_push_medium` | — | 20,000 |
| **仿真** | `lerobot/xarm_push_medium_replay` | — | 20,000 |
| **真实** | `lerobot/berkeley_autolab_ur5` | UC Berkeley | 97,939 |
| **真实** | `lerobot/columbia_cairlab_pusht_real` | Columbia | 27,808 |
| **真实** | `lerobot/nyu_door_opening_surprising_effectiveness` | NYU | 20,405 |
| | **合计** | | **251,802** |

> 亮点：`pusht` 有仿真版和真实版 → 天然的 **Sim vs Real** 对比实验

In [1]:
# =====================
# Step 0: 安装依赖（只需运行一次）
# =====================
!pip install datasets pyarrow pandas tqdm -q
print('依赖安装完成')

依赖安装完成


In [2]:
# =====================
# Step 1: 配置路径
# Colab: 挂载 Google Drive
# 本地: 直接用相对路径
# =====================
import os

# 检测是否在 Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/bigdata-project'
    print('Colab 环境，使用 Google Drive')
except ImportError:
    SAVE_DIR = '../data'
    print('本地环境，使用本地 ../data 目录')

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'数据将保存到: {SAVE_DIR}')

本地环境，使用本地 ../data 目录
数据将保存到: ../data


In [3]:
# =====================
# Step 2: 定义 8 个数据集
# =====================

# 仿真数据集
SIM_DATASETS = [
    'lerobot/pusht',
    'lerobot/xarm_lift_medium',
    'lerobot/xarm_lift_medium_replay',
    'lerobot/xarm_push_medium',
    'lerobot/xarm_push_medium_replay',
]

# 真实机器人数据集（三所大学：Berkeley / Columbia / NYU）
REAL_DATASETS = [
    'lerobot/berkeley_autolab_ur5',
    'lerobot/columbia_cairlab_pusht_real',
    'lerobot/nyu_door_opening_surprising_effectiveness',
]

ALL_DATASETS = SIM_DATASETS + REAL_DATASETS

print(f'共 {len(ALL_DATASETS)} 个数据集：')
for name in ALL_DATASETS:
    print(f'  {name}')

共 8 个数据集：
  lerobot/pusht
  lerobot/xarm_lift_medium
  lerobot/xarm_lift_medium_replay
  lerobot/xarm_push_medium
  lerobot/xarm_push_medium_replay
  lerobot/berkeley_autolab_ur5
  lerobot/columbia_cairlab_pusht_real
  lerobot/nyu_door_opening_surprising_effectiveness


In [4]:
# =====================
# Step 3: 先下载一个数据集，验证结构
# =====================
from datasets import load_dataset
import pandas as pd
import numpy as np

print('验证数据结构（以 pusht 为例）...')
ds_sample = load_dataset('lerobot/pusht', split='train')

print(f'\n数据集大小: {len(ds_sample):,} 行')
print(f'列名: {ds_sample.column_names}')

# 看第一行
row0 = ds_sample[0]
print(f'\n第一行示例：')
print(f'  episode_index: {row0["episode_index"]}')
print(f'  frame_index:   {row0["frame_index"]}')
print(f'  observation.state: shape={len(row0["observation.state"])}D, values={row0["observation.state"]}')
print(f'  action:        shape={len(row0["action"])}D, values={row0["action"]}')
print(f'  next.reward:   {row0["next.reward"]}')
print(f'  next.done:     {row0["next.done"]}')

验证数据结构（以 pusht 为例）...

数据集大小: 25,650 行
列名: ['observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index']

第一行示例：
  episode_index: 0
  frame_index:   0
  observation.state: shape=2D, values=[222.0, 97.0]
  action:        shape=2D, values=[233.0, 71.0]
  next.reward:   0.19029748439788818
  next.done:     False


In [5]:
# =====================
# Step 4: 各数据集维度说明
#
# 不同数据集的 state/action 维度不同：
#   - pusht (仿真): state=2D, action=2D
#   - xarm  (仿真): state=4D, action=3-4D
#   - 真实机器人:   state=8D, action=7D
#
# 解决方案：存原始数组，特征工程时统一用
#   L2 模长（维度无关的聚合方式）
# =====================

print('各数据集维度预览（快速抽样，每个取第1行）：')
print(f'{"数据集":50s} {"state维度":10s} {"action维度":10s} {"reward范围":15s}')
print('-' * 90)

for name in ALL_DATASETS:
    try:
        ds = load_dataset(name, split='train')
        row = ds[0]
        state_dim  = len(row['observation.state'])
        action_dim = len(row['action'])
        
        # 快速估算 reward 范围（取前100行）
        rewards = [ds[i]['next.reward'] for i in range(min(100, len(ds)))]
        r_min, r_max = min(rewards), max(rewards)
        
        short = name.split('/')[-1]
        print(f'{short:50s} {state_dim:10d} {action_dim:10d} [{r_min:.3f}, {r_max:.3f}]')
    except Exception as e:
        print(f'{name:50s} 加载失败: {e}')

各数据集维度预览（快速抽样，每个取第1行）：
数据集                                                state维度    action维度   reward范围       
------------------------------------------------------------------------------------------
pusht                                                       2          2 [0.174, 0.547]
xarm_lift_medium                                            4          4 [-0.033, 1.179]
xarm_lift_medium_replay                                     4          4 [-0.014, -0.003]
xarm_push_medium                                            4          3 [-0.979, -0.050]
xarm_push_medium_replay                                     4          3 [-1.278, -0.382]
berkeley_autolab_ur5                                        8          7 [0.000, 1.000]
columbia_cairlab_pusht_real                                 8          7 [0.000, 0.000]
nyu_door_opening_surprising_effectiveness                   8          7 [0.000, 1.000]


In [6]:
# =====================
# Step 5: 下载所有数据集并转为 DataFrame
#
# 每个数据集保存为独立 parquet，方便后续按需加载
# 同时保存合并版（全量帧）
# =====================
from tqdm import tqdm

def dataset_to_df(name):
    """将 HuggingFace dataset 转为 pandas DataFrame"""
    ds = load_dataset(name, split='train')
    short = name.split('/')[-1]
    
    rows = []
    for row in tqdm(ds, desc=f'  {short}', leave=False):
        rows.append({
            'source':          short,
            'episode_index':   row['episode_index'],
            'frame_index':     row['frame_index'],
            # 存为 list（维度因数据集而异）
            'observation_state': list(row['observation.state']),
            'action':            list(row['action']),
            'next_reward':     float(row['next.reward']),
            'next_done':       bool(row['next.done']),
        })
    
    return pd.DataFrame(rows)

all_frames = []

print('开始下载所有数据集...')
for name in ALL_DATASETS:
    short = name.split('/')[-1]
    save_path = f'{SAVE_DIR}/{short}.parquet'
    
    if os.path.exists(save_path):
        print(f'[跳过] {short} 已存在')
        df = pd.read_parquet(save_path)
    else:
        print(f'[下载] {name}')
        df = dataset_to_df(name)
        df.to_parquet(save_path, index=False)
        size_mb = os.path.getsize(save_path) / 1024 / 1024
        print(f'  → 保存: {save_path} ({len(df):,} 行, {size_mb:.1f} MB)')
    
    all_frames.append(df)

print('\n所有数据集下载完毕！')

开始下载所有数据集...
[跳过] pusht 已存在
[跳过] xarm_lift_medium 已存在
[跳过] xarm_lift_medium_replay 已存在
[跳过] xarm_push_medium 已存在
[跳过] xarm_push_medium_replay 已存在
[跳过] berkeley_autolab_ur5 已存在
[跳过] columbia_cairlab_pusht_real 已存在
[跳过] nyu_door_opening_surprising_effectiveness 已存在

所有数据集下载完毕！


In [7]:
# =====================
# Step 6: 合并所有帧，保存为 robot_frames.parquet
# =====================

robot_frames = pd.concat(all_frames, ignore_index=True)

combined_path = f'{SAVE_DIR}/robot_frames.parquet'
robot_frames.to_parquet(combined_path, index=False)

size_mb = os.path.getsize(combined_path) / 1024 / 1024

print('=== 合并完成 ===')
print(f'总帧数:   {len(robot_frames):,}')
print(f'总列数:   {len(robot_frames.columns)}')
print(f'文件大小: {size_mb:.1f} MB')
print(f'保存路径: {combined_path}')
print()

# 各数据集帧数
print('各数据集帧数：')
print(robot_frames.groupby('source').size().rename('帧数').to_string())

=== 合并完成 ===
总帧数:   251,802
总列数:   7
文件大小: 11.9 MB
保存路径: ../data/robot_frames.parquet

各数据集帧数：
source
berkeley_autolab_ur5                         97939
columbia_cairlab_pusht_real                  27808
nyu_door_opening_surprising_effectiveness    20405
pusht                                        25650
xarm_lift_medium                             20000
xarm_lift_medium_replay                      20000
xarm_push_medium                             20000
xarm_push_medium_replay                      20000


In [8]:
# =====================
# Step 7: 验证数据完整性
# =====================

print('=== 数据完整性检查 ===')
print()

# 1. 总行数
assert len(robot_frames) >= 100_000, f'帧数不足: {len(robot_frames):,}'
print(f'✅ 总帧数 {len(robot_frames):,} ≥ 100,000（满足课程要求）')

# 2. 必须列
required_cols = ['source', 'episode_index', 'frame_index',
                 'observation_state', 'action', 'next_reward', 'next_done']
for col in required_cols:
    assert col in robot_frames.columns, f'缺少列: {col}'
print(f'✅ 所有必须列存在: {required_cols}')

# 3. 无空值
null_count = robot_frames[['source','episode_index','frame_index',
                            'next_reward','next_done']].isnull().sum().sum()
assert null_count == 0, f'存在空值: {null_count}'
print(f'✅ 无空值')

# 4. 各数据集均存在
found_sources = set(robot_frames['source'].unique())
expected_short = {n.split('/')[-1] for n in ALL_DATASETS}
missing = expected_short - found_sources
assert len(missing) == 0, f'缺少数据集: {missing}'
print(f'✅ 8 个数据集全部加载成功')

# 5. reward 是数值
assert robot_frames['next_reward'].dtype in [float, 'float32', 'float64'], \
    'reward 不是数值类型'
print(f'✅ reward 是数值类型')

print()
print('所有验证通过！数据可以使用。')

=== 数据完整性检查 ===

✅ 总帧数 251,802 ≥ 100,000（满足课程要求）
✅ 所有必须列存在: ['source', 'episode_index', 'frame_index', 'observation_state', 'action', 'next_reward', 'next_done']
✅ 无空值
✅ 8 个数据集全部加载成功
✅ reward 是数值类型

所有验证通过！数据可以使用。


In [9]:
# =====================
# Step 8: 数据概览
# =====================

print('=== 数据集统计 ===')
print()

summary = robot_frames.groupby('source').agg(
    帧数=('episode_index', 'count'),
    episode数=('episode_index', 'nunique'),
    reward_min=('next_reward', 'min'),
    reward_max=('next_reward', 'max'),
    reward_mean=('next_reward', 'mean'),
).round(4)

print(summary.to_string())
print()
print('注意：xarm_push 系列的 reward 为负数（距离度量），范围不同')
print('→ 质量标签使用每个数据集自己的中位数作为阈值（不用固定值）')

=== 数据集统计 ===

                                              帧数  episode数  reward_min  reward_max  reward_mean
source                                                                                         
berkeley_autolab_ur5                       97939      1000      0.0000      1.0000       0.0102
columbia_cairlab_pusht_real                27808       136      0.0000      1.0000       0.0049
nyu_door_opening_surprising_effectiveness  20405       484      0.0000      1.0000       0.0237
pusht                                      25650       206      0.0000      0.9489       0.2914
xarm_lift_medium                           20000       800     -0.0527      1.4323       0.6171
xarm_lift_medium_replay                    20000       800     -0.0499      1.4318       0.6852
xarm_push_medium                           20000       800     -1.7147     -0.0431      -0.1923
xarm_push_medium_replay                    20000       800     -1.7464     -0.0509      -0.4736

注意：xarm_push 系列的 reward 

In [10]:
# =====================
# Step 9: 输出组员使用说明
# =====================

print('=' * 60)
print('告知组员的信息（发到群里）')
print('=' * 60)
print()
print('数据已下载完毕，共 251,802 帧，来自 8 个数据集。')
print()
print('加载代码：')
print()  
print('  from google.colab import drive')
print('  drive.mount("/content/drive")')
print('  import pandas as pd')
print('  df = pd.read_parquet("/content/drive/MyDrive/bigdata-project/robot_frames.parquet")')
print('  print(f"加载成功: {df.shape}")')
print()
print('列名说明：')
print('  source           → 数据集名称（哪个机器人/任务）')
print('  episode_index    → 第几次任务尝试')
print('  frame_index      → 该次任务的第几帧')
print('  observation_state→ 机器臂状态（list，维度因数据集而异）')
print('  action           → 动作指令（list，维度因数据集而异）')
print('  next_reward      → 当前帧得分')
print('  next_done        → 是否最后一帧')

告知组员的信息（发到群里）

数据已下载完毕，共 251,802 帧，来自 8 个数据集。

加载代码：

  from google.colab import drive
  drive.mount("/content/drive")
  import pandas as pd
  df = pd.read_parquet("/content/drive/MyDrive/bigdata-project/robot_frames.parquet")
  print(f"加载成功: {df.shape}")

列名说明：
  source           → 数据集名称（哪个机器人/任务）
  episode_index    → 第几次任务尝试
  frame_index      → 该次任务的第几帧
  observation_state→ 机器臂状态（list，维度因数据集而异）
  action           → 动作指令（list，维度因数据集而异）
  next_reward      → 当前帧得分
  next_done        → 是否最后一帧


---
## 组员：如何加载已下载的数据

Alex 下载完成后，组员只需要运行以下代码：

```python
# 1. 挂载 Drive（需要 Alex 共享文件夹权限）
from google.colab import drive
drive.mount('/content/drive')

# 2. 读取合并后的全量数据
import pandas as pd
df = pd.read_parquet('/content/drive/MyDrive/bigdata-project/robot_frames.parquet')
print(f'数据加载成功: {df.shape}')  # 应显示 (251802, 7)

# 3. 也可以只读某个数据集
pusht = pd.read_parquet('/content/drive/MyDrive/bigdata-project/pusht.parquet')
```

**不需要自己下载，不需要等待，直接开始分析！**